# Análise de dados TCP-CII

In [34]:
import pandas as pd

In [35]:
df = pd.read_csv('./T CELL/DENV 4 - T Cell Prediction - Class II.csv')
df

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_el core,netmhciipan_el score,netmhciipan_el percentile
0,1,KNQTWQIEKASLIEVK,206,221,16,HLA-DRB1*01:01,246,0.01,WQIEKASLI,0.986066,0.01
1,1,KNQTWQIEKASLIEVKT,206,222,17,HLA-DRB1*01:01,314,0.04,WQIEKASLI,0.978820,0.04
2,1,KNQTWQIEKASLIEV,206,220,15,HLA-DRB1*01:01,178,0.06,WQIEKASLI,0.974156,0.06
3,1,KNQTWQIEKASLIEVKTC,206,223,18,HLA-DRB1*01:01,382,0.06,WQIEKASLI,0.961404,0.06
4,1,WIESSKNQTWQIEKASLIEVK,201,221,21,HLA-DRB1*01:01,582,0.07,WQIEKASLI,0.829909,0.07
...,...,...,...,...,...,...,...,...,...,...,...
16411,1,TTASGKLVTQWCCR,301,314,14,HLA-DRB3*02:02,129,100.00,SGKLVTQWC,0.000011,100.00
16412,1,TTASGKLVTQWCCRSCTMPP,301,320,20,HLA-DRB3*02:02,535,100.00,LVTQWCCRS,0.000011,100.00
16413,1,KLVTQWCCRSCTMPPLRFLGE,306,326,21,HLA-DRB1*04:01,603,100.00,CRSCTMPPL,0.000010,100.00
16414,1,TTASGKLVTQWCC,301,313,13,HLA-DRB3*02:02,61,100.00,TASGKLVTQ,0.000006,100.00


## Selecionando Epítopos com median binding percentile menor que 5.

In [36]:
df_mbp_m5 = df[df['median binding percentile'] < 5].copy()
df_mbp_m5

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_el core,netmhciipan_el score,netmhciipan_el percentile
0,1,KNQTWQIEKASLIEVK,206,221,16,HLA-DRB1*01:01,246,0.01,WQIEKASLI,0.986066,0.01
1,1,KNQTWQIEKASLIEVKT,206,222,17,HLA-DRB1*01:01,314,0.04,WQIEKASLI,0.978820,0.04
2,1,KNQTWQIEKASLIEV,206,220,15,HLA-DRB1*01:01,178,0.06,WQIEKASLI,0.974156,0.06
3,1,KNQTWQIEKASLIEVKTC,206,223,18,HLA-DRB1*01:01,382,0.06,WQIEKASLI,0.961404,0.06
4,1,WIESSKNQTWQIEKASLIEVK,201,221,21,HLA-DRB1*01:01,582,0.07,WQIEKASLI,0.829909,0.07
...,...,...,...,...,...,...,...,...,...,...,...
575,1,HRLMSAAIKDQKAVHADMGYW,181,201,21,HLA-DQA1*01:02/DQB1*06:02,578,4.90,SAAIKDQKA,0.089729,4.90
576,1,QYKFQPESPARLASAILNAHK,31,51,21,HLA-DQA1*01:02/DQB1*06:02,548,4.90,ESPARLASA,0.089097,4.90
577,1,AKIFTPEARNSTFLIDGPD,121,139,19,HLA-DPA1*02:01/DPB1*01:01,432,4.90,ARNSTFLID,0.050604,4.90
578,1,SECPNERRAWNSLEVEDYGFG,141,161,21,HLA-DPA1*02:01/DPB1*01:01,570,4.90,WNSLEVEDY,0.029029,4.90


## Agrupando por pepitideos e agregando colunas pertinentes.

In [37]:
epitopos_repetidos = (
    df_mbp_m5
    .groupby('peptide', as_index=False)
    .agg(
        start=("start", "first"),
        end=("end", "first"),
        qte_de_alelos=("allele", "nunique"),
        median_binding_percentile=(
            "median binding percentile",
            "median"
        ),
        alelos=(
            "allele",
            lambda x: ", ".join(sorted(x.unique()))
        )
    ))

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AAIKDQKAVHADM,186,198,5,3.800,"HLA-DQA1*04:01/DQB1*04:02, HLA-DRB1*01:01, HLA..."
1,AAIKDQKAVHADMG,186,199,4,2.400,"HLA-DQA1*04:01/DQB1*04:02, HLA-DRB1*01:01, HLA..."
2,AAIKDQKAVHADMGY,186,200,2,1.700,"HLA-DRB1*01:01, HLA-DRB4*01:01"
3,AAIKDQKAVHADMGYW,186,201,2,2.800,"HLA-DRB1*01:01, HLA-DRB4*01:01"
4,AAIKDQKAVHADMGYWI,186,202,2,3.950,"HLA-DRB1*01:01, HLA-DRB4*01:01"
...,...,...,...,...,...,...
237,YRQGYATQTVGPWHLGK,256,272,3,2.000,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
238,YRQGYATQTVGPWHLGKL,256,273,2,2.055,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
239,YRQGYATQTVGPWHLGKLE,256,274,1,0.660,HLA-DQA1*05:01/DQB1*02:01
240,YRQGYATQTVGPWHLGKLEI,256,275,1,0.820,HLA-DQA1*05:01/DQB1*02:01


# Filtragem por epítopos que presentes em mais de 2 alelos.

In [38]:
epitopos_repetidos = epitopos_repetidos[
    epitopos_repetidos["qte_de_alelos"] >= 2
].reset_index(drop=True)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AAIKDQKAVHADM,186,198,5,3.800,"HLA-DQA1*04:01/DQB1*04:02, HLA-DRB1*01:01, HLA..."
1,AAIKDQKAVHADMG,186,199,4,2.400,"HLA-DQA1*04:01/DQB1*04:02, HLA-DRB1*01:01, HLA..."
2,AAIKDQKAVHADMGY,186,200,2,1.700,"HLA-DRB1*01:01, HLA-DRB4*01:01"
3,AAIKDQKAVHADMGYW,186,201,2,2.800,"HLA-DRB1*01:01, HLA-DRB4*01:01"
4,AAIKDQKAVHADMGYWI,186,202,2,3.950,"HLA-DRB1*01:01, HLA-DRB4*01:01"
...,...,...,...,...,...,...
114,YRQGYATQTVGPWH,256,269,4,1.625,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*04:01/DQB1..."
115,YRQGYATQTVGPWHL,256,270,3,1.400,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
116,YRQGYATQTVGPWHLG,256,271,4,2.550,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*04:01/DQB1..."
117,YRQGYATQTVGPWHLGK,256,272,3,2.000,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."


In [39]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["median_binding_percentile", "qte_de_alelos"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,YRQGYATQTVGPW,256,268,3,0.600,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
1,HRLMSAAIKDQKA,181,193,2,0.755,"HLA-DPA1*02:01/DPB1*14:01, HLA-DQA1*01:02/DQB1..."
2,AKIFTPEARNSTF,121,133,3,1.400,"HLA-DRB1*08:02, HLA-DRB1*11:01, HLA-DRB1*13:02"
3,YRQGYATQTVGPWHL,256,270,3,1.400,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
4,HRLMSAAIKDQKAVHADMGY,181,200,3,1.500,"HLA-DQA1*01:02/DQB1*06:02, HLA-DRB1*01:01, HLA..."
...,...,...,...,...,...,...
114,SECPNERRAWNSLEVEDY,141,158,2,4.250,"HLA-DQA1*01:01/DQB1*05:01, HLA-DQA1*05:01/DQB1..."
115,QYKFQPESPARLASAILNAH,31,50,3,4.300,"HLA-DPA1*02:01/DPB1*14:01, HLA-DQA1*01:02/DQB1..."
116,GIRSTTRLENVMWKQITNEL,56,75,2,4.350,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
117,GDVKGVLTKGKRALTPPVSDL,91,111,2,4.400,"HLA-DPA1*02:01/DPB1*14:01, HLA-DRB1*13:02"


In [40]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["median_binding_percentile", "qte_de_alelos"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,YRQGYATQTVGPW,256,268,3,0.600,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
1,HRLMSAAIKDQKA,181,193,2,0.755,"HLA-DPA1*02:01/DPB1*14:01, HLA-DQA1*01:02/DQB1..."
2,AKIFTPEARNSTF,121,133,3,1.400,"HLA-DRB1*08:02, HLA-DRB1*11:01, HLA-DRB1*13:02"
3,YRQGYATQTVGPWHL,256,270,3,1.400,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
4,HRLMSAAIKDQKAVHADMGY,181,200,3,1.500,"HLA-DQA1*01:02/DQB1*06:02, HLA-DRB1*01:01, HLA..."
...,...,...,...,...,...,...
114,SECPNERRAWNSLEVEDY,141,158,2,4.250,"HLA-DQA1*01:01/DQB1*05:01, HLA-DQA1*05:01/DQB1..."
115,QYKFQPESPARLASAILNAH,31,50,3,4.300,"HLA-DPA1*02:01/DPB1*14:01, HLA-DQA1*01:02/DQB1..."
116,GIRSTTRLENVMWKQITNEL,56,75,2,4.350,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
117,GDVKGVLTKGKRALTPPVSDL,91,111,2,4.400,"HLA-DPA1*02:01/DPB1*14:01, HLA-DRB1*13:02"


## Separando epítopos e criando arquivo FASTA para IEDB analysis resource

In [41]:
pepitides = epitopos_repetidos.peptide

with open("./peptideos.fasta", "w") as f:
    for i, peptide in enumerate(pepitides, start=1):
        f.write(f">NP {i}\n")
        f.write(f"{peptide}\n")
        
pepitides

0              YRQGYATQTVGPW
1              HRLMSAAIKDQKA
2              AKIFTPEARNSTF
3            YRQGYATQTVGPWHL
4       HRLMSAAIKDQKAVHADMGY
               ...          
114       SECPNERRAWNSLEVEDY
115     QYKFQPESPARLASAILNAH
116     GIRSTTRLENVMWKQITNEL
117    GDVKGVLTKGKRALTPPVSDL
118      AKIFTPEARNSTFLIDGPD
Name: peptide, Length: 119, dtype: str

### Seqkit remove sequências proteicas contendo gaps e *.

In [42]:
!seqkit grep -s -v -r -p '[-*X?]' './Fastas/denv4_NS1_final.fasta' > DENV4_seq_filter_all.fasta

### Resultado IEDB analysis resource

In [43]:
conservacy_result = pd.read_csv('./ConservancyResult_tcell_2.csv')
conservacy_result

,Epitope #,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,View details
0,1,NP 1,YRQGYATQTVGPW,13,62.42% (93/149),84.62%,100.00%,NaN
1,2,NP 2,HRLMSAAIKDQKA,13,95.30% (142/149),76.92%,100.00%,NaN
2,3,NP 3,AKIFTPEARNSTF,13,71.81% (107/149),76.92%,100.00%,NaN
3,4,NP 4,YRQGYATQTVGPWHL,15,62.42% (93/149),86.67%,100.00%,NaN
4,5,NP 5,HRLMSAAIKDQKAVHADMGY,20,95.30% (142/149),85.00%,100.00%,NaN
...,...,...,...,...,...,...,...,...
114,115,NP 115,SECPNERRAWNSLEVEDY,18,29.53% (44/149),83.33%,100.00%,NaN
115,116,NP 116,QYKFQPESPARLASAILNAH,20,88.59% (132/149),80.00%,100.00%,NaN
116,117,NP 117,GIRSTTRLENVMWKQITNEL,20,83.89% (125/149),85.00%,100.00%,NaN
117,118,NP 118,GDVKGVLTKGKRALTPPVSDL,21,24.16% (36/149),71.43%,100.00%,NaN


### Merge da coluna qte_de_alelos ao dataframe conservacy_result

In [44]:
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "qte_de_alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("View details"))

conservacy_result

,Epitope #,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos
0,1,NP 1,YRQGYATQTVGPW,13,62.42% (93/149),84.62%,100.00%,3
1,2,NP 2,HRLMSAAIKDQKA,13,95.30% (142/149),76.92%,100.00%,2
2,3,NP 3,AKIFTPEARNSTF,13,71.81% (107/149),76.92%,100.00%,3
3,4,NP 4,YRQGYATQTVGPWHL,15,62.42% (93/149),86.67%,100.00%,3
4,5,NP 5,HRLMSAAIKDQKAVHADMGY,20,95.30% (142/149),85.00%,100.00%,3
...,...,...,...,...,...,...,...,...
114,115,NP 115,SECPNERRAWNSLEVEDY,18,29.53% (44/149),83.33%,100.00%,2
115,116,NP 116,QYKFQPESPARLASAILNAH,20,88.59% (132/149),80.00%,100.00%,3
116,117,NP 117,GIRSTTRLENVMWKQITNEL,20,83.89% (125/149),85.00%,100.00%,2
117,118,NP 118,GDVKGVLTKGKRALTPPVSDL,21,24.16% (36/149),71.43%,100.00%,2


### Gerando a coluna percent_match para filtrar os epitopos com percentagem de match maior que 50%

In [45]:
col = "Percent of protein sequence matches at identity <= 100%"

conservacy_result["percent_match"] = (
    conservacy_result[col]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .astype(float)
)

conservacy_result

,Epitope #,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,percent_match
0,1,NP 1,YRQGYATQTVGPW,13,62.42% (93/149),84.62%,100.00%,3,62.42
1,2,NP 2,HRLMSAAIKDQKA,13,95.30% (142/149),76.92%,100.00%,2,95.30
2,3,NP 3,AKIFTPEARNSTF,13,71.81% (107/149),76.92%,100.00%,3,71.81
3,4,NP 4,YRQGYATQTVGPWHL,15,62.42% (93/149),86.67%,100.00%,3,62.42
4,5,NP 5,HRLMSAAIKDQKAVHADMGY,20,95.30% (142/149),85.00%,100.00%,3,95.30
...,...,...,...,...,...,...,...,...,...
114,115,NP 115,SECPNERRAWNSLEVEDY,18,29.53% (44/149),83.33%,100.00%,2,29.53
115,116,NP 116,QYKFQPESPARLASAILNAH,20,88.59% (132/149),80.00%,100.00%,3,88.59
116,117,NP 117,GIRSTTRLENVMWKQITNEL,20,83.89% (125/149),85.00%,100.00%,2,83.89
117,118,NP 118,GDVKGVLTKGKRALTPPVSDL,21,24.16% (36/149),71.43%,100.00%,2,24.16


### Sort por percent_match e filtragem por percent_match >= 95.00

In [46]:
conservacy_result_filtered = (
    conservacy_result[conservacy_result["percent_match"] >= 95.0]
    .sort_values(
            by="percent_match", 
            ascending=False
        )
    ).reset_index(drop=True)

conservacy_result_filtered

,Epitope #,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,percent_match
0,9,NP 9,QYKFQPESPARLA,13,98.66% (147/149),84.62%,100.00%,13,98.66
1,18,NP 18,QYKFQPESPARLAS,14,98.66% (147/149),78.57%,100.00%,13,98.66
2,96,NP 96,PSLRTTTASGKLV,13,98.66% (147/149),92.31%,100.00%,2,98.66
3,15,NP 15,QYKFQPESPARLASA,15,98.66% (147/149),73.33%,100.00%,11,98.66
4,48,NP 48,QYKFQPESPARLASAI,16,97.99% (146/149),75.00%,100.00%,10,97.99
5,95,NP 95,ADMGYWIESSKNQT,14,97.99% (146/149),78.57%,100.00%,2,97.99
6,97,NP 97,HTWTEQYKFQPESPARLA,18,97.99% (146/149),88.89%,100.00%,7,97.99
7,113,NP 113,HTWTEQYKFQPESPARL,17,97.99% (146/149),88.24%,100.00%,4,97.99
8,114,NP 114,HTWTEQYKFQPESPAR,16,97.99% (146/149),93.75%,100.00%,2,97.99
9,49,NP 49,HTWTEQYKFQPESPARLASA,20,97.99% (146/149),80.00%,100.00%,9,97.99
